In [2]:
!pip install mamba-ssm[causal-conv1d] --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 3.2 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl size=193269254 sha256=407c0b75f40e845c006d04371c3d1bdf87b936af2ceb02f97cdb4c21749e167a
  Stored in directory: /root/.cache/pip/wheels/98/4a/75/b24971cff4599825b16b612f08fbd2e60a2c336a56e081a3c8
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533545871 sha256=8d079ac13497bcfd339f93aa06879d1830424d6b17e1fc404a83db61a42316e2
  Stored in directory: /root/.cache/pip/wheels/28/83/54/d45107838fec575b93f5d723f56351cee19a1b13bcd4ec9f3f
Successfully built causal-conv1d mamba-ssm


In [3]:
from mamba_ssm import Mamba
import torch

# Quick test
model = Mamba(d_model=128, d_state=16, d_conv=4, expand=2).cuda()
x = torch.randn(4, 60, 128).cuda()
y = model(x)
print(f"Input: {x.shape} → Output: {y.shape}")
print("✓ mamba-ssm CUDA kernel working!")

Input: torch.Size([4, 60, 128]) → Output: torch.Size([4, 60, 128])
✓ mamba-ssm CUDA kernel working!


In [4]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 1024**3:.2f} GB")
    print(f"  Allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")

GPUs: 2
GPU 0: Tesla T4, 14.56 GB
  Allocated: 0.01 GB
  Reserved:  0.02 GB
GPU 1: Tesla T4, 14.56 GB
  Allocated: 0.00 GB
  Reserved:  0.00 GB


In [5]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/siddharthamallavolu/bm-lite-v0-01")

import torch.nn as nn
from configs import BASE_CONFIG
from train import Trainer, TrainConfig

cfg = TrainConfig(
    dataset_root="/kaggle/input/datasets/siddharthamallavolu/musdb18hq",
    target_stem="vocals",
    batch_size=3,            # ← reduced from 6 (no AMP = more memory)
    chunk_duration=5.0,
    grad_accumulation=2,     # ← effective batch still = 6
    num_workers=4,
    lr=3e-4,
    use_amp=False,           # ← stable with mamba-ssm
    save_dir="/kaggle/working/",
    save_every=2,            # ← save more often for NaN recovery
    log_every=20,
    model_config=BASE_CONFIG,
)

trainer = Trainer(cfg)
trainer.model = nn.DataParallel(trainer.model)

for block in trainer.model.module.blocks:
    block.frequency.chunk_size = 48

trainer.train()


  BandMamba-Light Training Setup
  Device:          cuda
  Target stem:     vocals
  Epochs:          100 (warmup: 20)
  Batch size:      3 (× 2 accumulation = 6 effective)
  Learning rate:   0.0003
  Grad clip norm:  1.0
  Mixed precision: False


  BandMamba-Light Parameter Summary
  stft_module                         0  (0.00M)
  encoder                       225,792  (0.23M)
  blocks                      2,855,424  (2.86M)
  mask_estimator                 33,280  (0.03M)
  decoders                    1,056,768  (1.06M)
  TOTAL                       4,171,264  (4.17M)
MUSDB18-HQ [train]: 100 tracks, target='vocals', chunk=5.0s, samples_per_track=10
MUSDB18-HQ [test]: 50 tracks, target='vocals', chunk=5.0s, samples_per_track=1

Starting training from epoch 0...

Epoch 1/100 [Phase 1: L1+STFT]
  [20/333]  loss: 2.0365  lr: 0.000300  time: 34.0s
  [40/333]  loss: 2.1263  lr: 0.000300  time: 58.3s
  [60/333]  loss: 2.0883  lr: 0.000300  time: 81.9s
  [80/333]  loss: 2.1105  lr: 0.0003

In [2]:
# Cell 6 (after training): Download best model
from IPython.display import FileLink
FileLink("/kaggle/working/best_model.pt")

/kaggle/working/best_model.pt